# 1단계 v3. 사건 단위 사칭 판정을 포함한 데이터셋 구축

> 보이스피싱 전사자료, 정상 금융상담, 우체국 피해사례를 구조화하여 17개 데이터테이블을 생성합니다.

이 노트북에서는 **데이터 생성과 기본 무결성 검사**만 수행합니다. 사람 표본 검수와 머신러닝 학습은 다음 단계에서 진행합니다.

영문 컬럼은 Python·ML 작업용으로 유지하고, `보이스피싱_사건요약_한글`에는 대시보드 파생변수까지 모두 한글 컬럼으로 저장합니다.

v3에서는 사칭기관을 단일 발화로 강제 선택하지 않고 사건 전체 후보를 비교하여 `HIGH·MEDIUM·UNKNOWN` 신뢰등급, 접근·주요·보조기관 및 사칭 전환순서를 추가합니다.

In [ ]:
# 0. 분석에 필요한 라이브러리 설치
!pip -q install pandas pyarrow openpyxl

In [ ]:
# 1. Google Drive 연결
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
print('Google Drive 연결 완료')

## 2. 경로와 입력 데이터 확인

정상상담 ZIP은 전체 12개 중 구조화 JSON이 있는 `TL_`, `VL_` 6개를 사용합니다.

In [ ]:
# 2. 파일 경로 설정
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / '보이스피싱_분석'
RESULT_ROOT = PROJECT_ROOT / '분석 결과'
DATASET_ROOT = RESULT_ROOT / '데이터셋'
STEP_ROOT = DATASET_ROOT / '01_dataset_building'
SCRIPT_PATH = STEP_ROOT / 'build_voicephishing_datasets_v3.py'
# 기존 v3 결과는 보존하고 사건 단위 사칭 판정 결과는 v4에 저장합니다.
OUTPUT_ROOT = PROJECT_ROOT / '구축 데이터셋_v4'

# 정상상담 폴더는 현재 구조와 이전 구조를 모두 확인합니다.
normal_candidates = [
    RESULT_ROOT / '00.금융분야 고객상담 데이터',
    PROJECT_ROOT / '00.금융분야 고객상담 데이터',
]
NORMAL_ROOT = next((path for path in normal_candidates if path.exists()), normal_candidates[0])

# df_postal.csv는 데이터셋 폴더 안에서 정확한 파일명을 찾습니다.
postal_candidates = list(DATASET_ROOT.rglob('df_postal.csv'))
POSTAL_PATH = postal_candidates[0] if postal_candidates else DATASET_ROOT / 'df_postal.csv'

print('분석 결과:', RESULT_ROOT)
print('정상상담:', NORMAL_ROOT)
print('우체국 자료:', POSTAL_PATH)
print('v3 스크립트:', SCRIPT_PATH)
print('저장 위치:', OUTPUT_ROOT)

In [ ]:
# 3. 입력 파일 수 확인
import pandas as pd

phishing_paths = list(RESULT_ROOT.rglob('cases.json'))
all_normal_zips = list(NORMAL_ROOT.rglob('*.zip')) if NORMAL_ROOT.exists() else []
labeled_normal_zips = [p for p in all_normal_zips if p.name.startswith(('TL_', 'VL_'))]

print('보이스피싱 cases.json:', len(phishing_paths), '개')
print('정상상담 전체 ZIP:', len(all_normal_zips), '개')
print('실제 처리할 TL/VL ZIP:', len(labeled_normal_zips), '개')
print('처리 ZIP:', [p.name for p in labeled_normal_zips])

assert phishing_paths, 'cases.json을 찾지 못했습니다.'
assert labeled_normal_zips, 'TL_/VL_ 정상상담 ZIP을 찾지 못했습니다.'
assert POSTAL_PATH.exists(), f'df_postal.csv를 찾지 못했습니다: {POSTAL_PATH}'
assert SCRIPT_PATH.exists(), f'v3 스크립트를 찾지 못했습니다: {SCRIPT_PATH}'

postal_preview_df = pd.read_csv(POSTAL_PATH, encoding='utf-8-sig')
print('우체국 피해사례:', len(postal_preview_df), '행')
display(postal_preview_df.head(3))

## 3. 구축 스크립트 확인

`old` 폴더의 구버전을 잘못 실행하지 않도록 v3 파일을 정확한 경로로 지정합니다.

In [ ]:
# 4. 실행할 스크립트 버전 확인
import re

script_text = SCRIPT_PATH.read_text(encoding='utf-8')
version_match = re.search(r'SCRIPT_VERSION\s*=\s*"([^"]+)"', script_text)
script_version = version_match.group(1) if version_match else 'UNKNOWN'

print('실행 스크립트:', SCRIPT_PATH.name)
print('스크립트 버전:', script_version)
assert script_version.startswith('3.'), 'v3 스크립트가 아닙니다.'

## 4. 17개 데이터테이블 생성

표준 8개, 머신러닝 준비 4개, 대시보드 5개를 생성합니다. 모델 학습은 하지 않습니다.

In [ ]:
# 5. 데이터셋 구축 실행
import subprocess
import sys

# v4를 처음 만들 때는 False를 유지합니다. 중단 후 같은 버전을 이어갈 때만 True로 바꿉니다.
SKIP_EXISTING = False

command = [
    sys.executable, str(SCRIPT_PATH),
    '--result-root', str(RESULT_ROOT),
    '--normal-root', str(NORMAL_ROOT),
    '--postal-path', str(POSTAL_PATH),
    '--output-root', str(OUTPUT_ROOT),
]
if SKIP_EXISTING:
    command.append('--skip-existing')

subprocess.run(command, check=True)
print('17개 데이터테이블 생성 명령 완료')

## 5. 생성 결과 확인

먼저 17개 표가 모두 생성됐는지 확인합니다.

In [ ]:
# 6. 데이터셋 목록 확인
import json

REPORT_ROOT = OUTPUT_ROOT / '04_reports'
manifest_path = REPORT_ROOT / 'dataset_manifest.json'
validation_path = REPORT_ROOT / 'validation_report.json'

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
table_manifest_df = pd.DataFrame(manifest['tables'])

print('스크립트 버전:', manifest['metadata']['script_version'])
print('입력 파일:', manifest['metadata']['input_counts'])
print('생성 테이블:', len(table_manifest_df), '개')
display(table_manifest_df[['name', 'rows', 'columns', 'csv', 'parquet']])

assert len(table_manifest_df) == 17, f'생성 테이블이 17개가 아닙니다: {len(table_manifest_df)}개'

## 6. 기본 무결성 검사

이 검사는 데이터가 정상적으로 생성·연결됐는지 확인합니다. 의미가 정확한지는 2단계에서 사람이 표본 검수합니다.

In [ ]:
# 7. ID 중복·누락·연결 오류 확인
validation = json.loads(validation_path.read_text(encoding='utf-8'))
summary = validation['integrity_summary']

print('ID 중복:', summary['duplicate_id_total'], '건')
print('필수 ID 누락:', summary['missing_id_total'], '건')
print('테이블 연결 오류:', summary['foreign_key_orphan_total'], '건')
print('비어 있는 필수 테이블:', summary['empty_critical_tables'])
print('기본 무결성 통과:', summary['integrity_passed'])

assert summary['integrity_passed'], '기본 무결성 검사에 실패했습니다. 2단계로 넘어가면 안 됩니다.'
assert validation['vp_files']['rows'] == len(phishing_paths), '입력 JSON 수와 파일 테이블 행 수가 다릅니다.'
assert validation['normal_finance_calls']['rows'] > 0, '정상 금융상담 데이터가 생성되지 않았습니다.'
print('기본 무결성 검사 통과')

In [ ]:
# 8. CSV와 Parquet 저장 확인
missing_csv = table_manifest_df[~table_manifest_df['csv'].map(lambda path: Path(path).exists())]
missing_parquet = table_manifest_df[table_manifest_df['parquet'].isna() | ~table_manifest_df['parquet'].fillna('').map(lambda path: Path(path).exists() if path else False)]

print('CSV 누락:', len(missing_csv), '개')
print('Parquet 누락:', len(missing_parquet), '개')
assert missing_csv.empty, '저장되지 않은 CSV가 있습니다.'
assert missing_parquet.empty, '저장되지 않은 Parquet이 있습니다.'
print('CSV 17개 + Parquet 17개 저장 완료')

## 7. 주요 테이블 미리보기

In [ ]:
# 9. 분석에 자주 사용할 표 확인
STANDARD_ROOT = OUTPUT_ROOT / '01_standard_tables'
ML_ROOT = OUTPUT_ROOT / '02_ml_tables'
DASHBOARD_ROOT = OUTPUT_ROOT / '03_dashboard_tables'

preview_paths = {
    '사건 테이블': STANDARD_ROOT / 'vp_cases.parquet',
    '발화 테이블': STANDARD_ROOT / 'vp_utterances.parquet',
    '금액 이벤트': STANDARD_ROOT / 'vp_amount_events.parquet',
    '정상·사기 ML 준비표': ML_ROOT / 'fraud_detection_ml.parquet',
    '한글 사건 요약': DASHBOARD_ROOT / '보이스피싱_사건요약_한글.parquet',
}

for table_name, table_path in preview_paths.items():
    preview_df = pd.read_parquet(table_path)
    print(f'\n{table_name}: {len(preview_df):,}행 × {len(preview_df.columns):,}열')
    display(preview_df.head(3))

# 한글 사건요약에는 대시보드 분석 변수가 모두 한글로 표시됩니다.
korean_summary_df = pd.read_parquet(DASHBOARD_ROOT / '보이스피싱_사건요약_한글.parquet')
print('한글 사건요약 전체 컬럼 수:', len(korean_summary_df.columns))
print(korean_summary_df.columns.tolist())

# 사건 단위 사칭 판정 컬럼과 신뢰등급 확인
case_summary_df = pd.read_parquet(STANDARD_ROOT / 'vp_cases.parquet')
required_case_columns = {
    'access_impersonation_subtype',
    'primary_impersonation_subtype',
    'secondary_impersonation_subtypes',
    'mentioned_impersonation_subtypes',
    'explicit_claim_subtypes',
    'impersonation_transition_sequence',
    'primary_impersonation_confidence_tier',
    'impersonation_review_required',
}
assert required_case_columns.issubset(case_summary_df.columns), (
    '사건 단위 사칭 컬럼 누락: ' + str(required_case_columns - set(case_summary_df.columns))
)
confidence_summary_df = pd.crosstab(
    case_summary_df['supervised_target'],
    case_summary_df['primary_impersonation_confidence_tier'],
    margins=True
)
display(confidence_summary_df)
print('사건 단위 사칭 판정 컬럼 검증 완료')

## 실행 완료

다음 문구가 모두 확인되면 1단계가 완료된 것입니다.

```text
생성 테이블: 17개
기본 무결성 검사 통과
CSV 17개 + Parquet 17개 저장 완료
```

다음 단계에서는 기존 2단계 품질검사와 3단계 ML 노트북을 그대로 사용합니다. 새 데이터셋을 사용할 때는 각 노트북의 `DATASET_ROOT`만 `구축 데이터셋_v4`로 지정합니다.

주의: 사칭·요구행동·심리전략·금액 상태는 규칙 기반 `SILVER` 라벨이며, 실제 피해 여부를 의미하지 않습니다.